In [83]:
app_py_content = """
import streamlit as st
import pandas as pd
import joblib

# PAGE CONFIGURATION

st.set_page_config(
    page_title="Student Performance Prediction",
    page_icon="🎓",
    layout="centered"
)

# LOAD TRAINED MODEL

model = joblib.load("student_performance_model.pkl")

# load the features names used when training
features = joblib.load("student_features.pkl")

# TITLE

st.title("🎓 student Performance Prediction")

st.write(
    "This application uses Linear Regression "
    "to predict a student's overall score."
)

st.divider()

# STUDENT INPUT

st.subheader("Enter Student Information")


# study hours
study_hours = st.number_input(
    "Study Hours",
    min_value=0,
    max_value=24,
    value=0
)


# attendance percentage
attendance_percentage = st.number_input(
    "Attendance Percentage",
    min_value=0,
    max_value=100,
    value=75,
    step=0.1
)


# internet access
internet_access = st.selectbox(
    "Internet Access",
    options=["Yes", "No"]
)


# travel time
travel_time = st.selectbox(
    "Travel Time",
    [
        "<15 min",
        "15-30 min",
        "30-60 min",
        ">60 min"
    ]
)


# extra activities
extra_activities = st.selectbox(
    "Extra Activities",
    options=["Yes", "No"]
)


# PREDICTION BUTTON

if st.button("💡 Predict overall score"):

    # Create a DataFrame with the user input
    input_data = pd.DataFrame({
        "study_hours": [study_hours],
        "attendance_percentage": [attendance_percentage],
        "internet_access": [internet_access],
        "travel_time": [travel_time],
        "extra_activities": [extra_activities]
    })


    # ENCODE CATEGORICAL VARIABLES

    input_data = pd.get_dummies(
        input_data,
        columns=[
            "internet_access",
            "travel_time",
            "extra_activities"
        ],
        drop_first=True
    )


    # MATCH TRAINING COLUMNS

    input_data = input_data.reindex(
        columns=features,
        fill_value=0
    )


    # MAKE PREDICTION

    prediction = model.predict(input_data)[0]


    # keep prediction between 0 and 100
    prediction = max(0, min(100, prediction))


    # DISPLAY RESULT

    st.success(
        f"Prediction Overall Score: {prediction:.2f}"
    )


    # PERFORMANCE LEVEL

    if prediction >= 70:

        st.write("🌟 performance level: Excellent")

    elif prediction >= 50:

        st.write("👍 performance level: Average")

    else:

        st.write("📚 performance level: Needs Improvement")



    # DISPLAY INPUT

    st.subheader("Student information")

    st.dataframe(
        {
            "Study Hours": [study_hours],
            "Attendance (%)": [attendance_percentage],
            "Internet Access": [internet_access],
            "Travel Time": [travel_time],
            "Extra Activities": [extra_activities]
        }
    )


# ABOUT PROJECT

st.divider()

st.subheader("About This Project")

st.write(
    "This project uses Linear Regression to predict "
    "a student's overall score based on study hours "
    "attendance percentage, internet access, travel time, "
    "and participation in extra activities."
)

st.write(
    "Linear Regression was selected because the target"
    "variable, overall_score, is continuous and numerical."
)




    """

In [84]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import joblib

# load dataset

In [85]:
import pandas as pd

data = pd.read_csv("/content/Student_Performance.csv")

print("Dataset loaded successfully")

print(data.head())

Dataset loaded successfully
   student_id  age  gender school_type parent_education  study_hours  \
0           1   14    male      public    post graduate          3.1   
1           2   18  female      public         graduate          3.7   
2           3   17  female     private    post graduate          7.9   
3           4   16   other      public      high school          1.1   
4           5   16  female      public      high school          1.3   

   attendance_percentage internet_access travel_time extra_activities  \
0                   84.3             yes     <15 min              yes   
1                   87.8             yes     >60 min               no   
2                   65.5              no     <15 min               no   
3                   58.1              no   15-30 min               no   
4                   61.0             yes   30-60 min              yes   

  study_method  math_score  science_score  english_score  overall_score  \
0        notes        42.

In [86]:
print(data.columns.tolist())

['student_id', 'age', 'gender', 'school_type', 'parent_education', 'study_hours', 'attendance_percentage', 'internet_access', 'travel_time', 'extra_activities', 'study_method', 'math_score', 'science_score', 'english_score', 'overall_score', 'final_grade']


# **select features**

In [87]:
# Select the features you want to use as X

X = data[
    [
      "study_hours",
      "attendance_percentage",
      "internet_access",
      "travel_time",
      "extra_activities"
   ]
]
print(X.head())

   study_hours  attendance_percentage internet_access travel_time  \
0          3.1                   84.3             yes     <15 min   
1          3.7                   87.8             yes     >60 min   
2          7.9                   65.5              no     <15 min   
3          1.1                   58.1              no   15-30 min   
4          1.3                   61.0             yes   30-60 min   

  extra_activities  
0              yes  
1               no  
2               no  
3               no  
4              yes  


In [88]:
y = data["overall_score"]

print(y.head())

0    53.1
1    61.3
2    89.6
3    41.6
4    25.4
Name: overall_score, dtype: float64


In [89]:
# convert categorical feature into numerical values

x = pd.get_dummies(
    X,
    columns=[
        "internet_access",
        "travel_time",
        "extra_activities"
    ],
    drop_first=True
)

print(x.head())

   study_hours  attendance_percentage  internet_access_yes  \
0          3.1                   84.3                 True   
1          3.7                   87.8                 True   
2          7.9                   65.5                False   
3          1.1                   58.1                False   
4          1.3                   61.0                 True   

   travel_time_30-60 min  travel_time_<15 min  travel_time_>60 min  \
0                  False                 True                False   
1                  False                False                 True   
2                  False                 True                False   
3                  False                False                False   
4                   True                False                False   

   extra_activities_yes  
0                  True  
1                 False  
2                 False  
3                 False  
4                  True  


In [90]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data:, x_train.shape")
print("Testing data:, x_test.shape")

Training data:, x_train.shape
Testing data:, x_test.shape


# **LINEAR REGRESSION**

In [91]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(x_train, y_train)

print("Linear Regression model trained successfully")

# Create a linear regression model
model = LinearRegression()

# Train the model on the training data
model.fit(x_train, y_train)

Linear Regression model trained successfully


LinearRegression()

In [92]:
y_pred = model.predict(x_test)

print("predicted values:")
print(y_pred[:10])


predicted values:
[83.19096203 47.29892687 63.08885427 44.08150201 44.56137792 79.29124839
 37.14029481 65.26076263 53.58154445 75.76829167]


In [93]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

mae = mean_absolute_error(y_test, y_pred)

mse = mean_squared_error(y_test, y_pred)

rmse = mse ** 0.5

r2 = r2_score(y_test, y_pred)

print("Mean Absolute Error:", mae)
print("Moean Squared Error:", mse)
print("Root mean Squared Error:", rmse)
print("R2 Score:", r2)

Mean Absolute Error: 5.009785646365541
Moean Squared Error: 33.38069122581406
Root mean Squared Error: 5.777602550004116
R2 Score: 0.9080602164347146


In [94]:
import joblib

joblib.dump(model, "student_performance_model.pkl")

joblib.dump(x.columns.tolist(), "student_features.pkl")

print("Model saved successfully")
print("Features saved successfully")



Model saved successfully
Features saved successfully


# **app.py**

In [95]:
%%writefile app.py

import streamlit as st
import pandas as pd
import joblib

# PAGE CONFIGURATION

st.set_page_config(
    page_title="Student Performance Prediction",
    page_icon="🎓",
    layout="centered"
)

# LOAD TRAINED MODEL

model = joblib.load("student_performance_model.pkl")

# load the features names used when training
features = joblib.load("student_features.pkl")

# TITLE

st.title("🎓 student Performance Prediction")

st.write(
    "This application uses Linear Regression "
    "to predict a student's overall score."
)

st.divider()

# STUDENT INPUT

st.subheader("Enter Student Information")


# study hours
study_hours = st.number_input(
    "Study Hours",
    min_value=0,
    max_value=24,
    value=0
)


# attendance percentage
attendance_percentage = st.number_input(
    "Attendance Percentage",
    min_value=0,
    max_value=100,
    value=75,
    step=0.1
)


# internet access
internet_access = st.selectbox(
    "Internet Access",
    options=["Yes", "No"]
)


# travel time
travel_time = st.selectbox(
    "Travel Time",
    [
        "<15 min",
        "15-30 min",
        "30-60 min",
        ">60 min"
    ]
)


# extra activities
extra_activities = st.selectbox(
    "Extra Activities",
    options=["Yes", "No"]
)


# PREDICTION BUTTON

if st.button("💡 Predict overall score"):

    # Create a DataFrame with the user input
    input_data = pd.DataFrame({
        "study_hours": [study_hours],
        "attendance_percentage": [attendance_percentage],
        "internet_access": [internet_access],
        "travel_time": [travel_time],
        "extra_activities": [extra_activities]
    })


    # ENCODE CATEGORICAL VARIABLES

    input_data = pd.get_dummies(
        input_data,
        columns=[
            "internet_access",
            "travel_time",
            "extra_activities"
        ],
        drop_first=True
    )


    # MATCH TRAINING COLUMNS

    input_data = input_data.reindex(
        columns=features,
        fill_value=0
    )


    # MAKE PREDICTION

    prediction = model.predict(input_data)[0]


    # keep prediction between 0 and 100
    prediction = max(0, min(100, prediction))


    # DISPLAY RESULT

    st.success(
        f"Prediction Overall Score: {prediction:.2f}"
    )


    # PERFORMANCE LEVEL

    if prediction >= 70:

        st.write("🌟 performance level: Excellent")

    elif prediction >= 50:

        st.write("👍 performance level: Average")

    else:

        st.write("📚 performance level: Needs Improvement")



    # DISPLAY INPUT

    st.subheader("Student information")

    st.dataframe(
        {
            "Study Hours": [study_hours],
            "Attendance (%)": [attendance_percentage],
            "Internet Access": [internet_access],
            "Travel Time": [travel_time],
            "Extra Activities": [extra_activities]
        }
    )


# ABOUT PROJECT

st.divider()

st.subheader("About This Project")

st.write(
    "This project uses Linear Regression to predict "
    "a student's overall score based on study hours "
    "attendance percentage, internet access, travel time, "
    "and participation in extra activities."
)

st.write(
    "Linear Regression was selected because the target"
    "variable, overall_score, is continuous and numerical."
)



Overwriting app.py


In [96]:
import os

print(os.path.exists("app.py"))

True


# **GENERATE REQUIREMENT**

In [97]:
%%writefile requirements.txt
streamlit
pandas
scikit-learn
joblib


Overwriting requirements.txt
